# Laboratoire 2 : Classifieur classique (baseline)

**Quantum Machine Learning — Master/PhD**

Semaine 1 — Fondements de l'apprentissage automatique

## Objectifs

- Implémenter un pipeline de Machine Learning classique complet
- Établir les métriques de baseline pour tous les benchmarks QML
- Maîtriser SVM, Random Forest, MLP avec scikit-learn et PyTorch
- Comprendre l'importance du feature scaling, de la cross-validation et des métriques

## Bibliothèques requises

- `scikit-learn` (≥ 1.x)
- `torch` (≥ 2.x)
- `matplotlib`
- `numpy`
- `seaborn`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print('Bibliothèques chargées avec succès.')
print(f'PyTorch version : {torch.__version__}')

---
## Partie A : SVM avec noyaux RBF et polynomial

Le SVM (Support Vector Machine) classifie en trouvant l'hyperplan séparateur optimal.
Le *kernel trick* permet de projeter dans un espace de dimension supérieure.

In [ ]:
# Chargement et préparation des données Iris

iris = datasets.load_iris()
X, y = iris.data, iris.target

# Pour simplifier la visualisation, on garde 2 classes
mask = y != 2
X_bin, y_bin = X[mask], y[mask]

print(f"Iris dataset : {X.shape[0]} échantillons, {X.shape[1]} features")
print(f"Classes : {iris.target_names}")
print(f"Features : {iris.feature_names}")
print(f"\nClassification binaire (classes 0 et 1) : {X_bin.shape[0]} échantillons")

In [ ]:
# SVM avec noyau RBF et polynomial

X_train, X_test, y_train, y_test = train_test_split(
    X_bin, y_bin, test_size=0.3, random_state=42, stratify=y_bin
)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SVM RBF
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)

# SVM Polynomial
svm_poly = SVC(kernel='poly', degree=3, C=1.0, gamma='scale', probability=True)
svm_poly.fit(X_train_scaled, y_train)
y_pred_poly = svm_poly.predict(X_test_scaled)

# Résultats
print("=== SVM RBF ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_rbf):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred_rbf):.4f}")

print("\n=== SVM Polynomial (deg=3) ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_poly):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred_poly):.4f}")

In [ ]:
# Optimisation des hyperparamètres avec GridSearchCV

param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 'auto', 0.1, 1.0],
    'kernel': ['rbf', 'poly'],
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train_scaled, y_train)

print("Meilleurs hyperparamètres :")
print(grid.best_params_)
print(f"Meilleur score CV : {grid.best_score_:.4f}")
print(f"Test accuracy     : {grid.score(X_test_scaled, y_test):.4f}")

---
## Partie B : Random Forest et MLP

Comparaison de différents classifieurs classiques.

In [ ]:
# Random Forest

rf = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42
)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

# MLP (scikit-learn)
mlp_sk = MLPClassifier(
    hidden_layer_sizes=(50, 25),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)
mlp_sk.fit(X_train_scaled, y_train)
y_pred_mlp = mlp_sk.predict(X_test_scaled)

# Comparaison
modeles = {
    'SVM RBF (best)': grid.best_estimator_,
    'Random Forest': rf,
    'MLP (scikit-learn)': mlp_sk,
}

print(f"{'Modèle':<25} {'Accuracy':<12} {'F1-score':<12}")
print('-' * 49)
for nom, modele in modeles.items():
    y_pred = modele.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"{nom:<25} {acc:<12.4f} {f1:<12.4f}")

In [ ]:
# MLP avec PyTorch

class MLP_torch(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=50, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim),
        )

    def forward(self, x):
        return self.net(x)

# Conversion en tenseurs
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

model = MLP_torch(input_dim=X.shape[1], hidden_dim=50, output_dim=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Entraînement
epochs = 100
losses = []
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 20 == 0:
        print(f"Époque {epoch+1:3d}/{epochs} — Loss : {losses[-1]:.4f}")

# Évaluation
with torch.no_grad():
    outputs = model(X_test_t)
    _, y_pred_torch = torch.max(outputs, 1)
    acc_torch = accuracy_score(y_test, y_pred_torch.numpy())
    print(f"\nMLP PyTorch — Test accuracy : {acc_torch:.4f}")

# Courbe d'apprentissage
plt.figure(figsize=(6, 4))
plt.plot(losses, label='Training loss')
plt.xlabel('Époque')
plt.ylabel('Cross-entropy loss')
plt.title('Courbe d\'apprentissage — MLP PyTorch')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

---
## Partie C : Métriques avancées et cross-validation

Au-delà de l'accuracy, on utilise F1-score, ROC-AUC, et la validation croisée.

In [ ]:
# ROC-AUC et courbes ROC

# Probabilités prédites
y_prob_rbf = grid.best_estimator_.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

auc_rbf = roc_auc_score(y_test, y_prob_rbf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"AUC SVM RBF   : {auc_rbf:.4f}")
print(f"AUC Random Forest : {auc_rf:.4f}")

# Courbes ROC
fpr_rbf, tpr_rbf, _ = roc_curve(y_test, y_prob_rbf)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(6, 5))
plt.plot(fpr_rbf, tpr_rbf, label=f'SVM RBF (AUC = {auc_rbf:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aléatoire')
plt.xlabel('Taux de faux positifs (FPR)')
plt.ylabel('Taux de vrais positifs (TPR)')
plt.title('Courbes ROC')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Cross-validation

print("=== Cross-validation (5-fold) ===")

for nom, modele in modeles.items():
    scores = cross_val_score(modele, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{nom:<25} — Accuracy CV: {scores.mean():.4f} ± {scores.std():.4f}")

# Matrice de confusion
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nom, modele) in zip(axes, modeles.items()):
    y_pred = modele.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{nom}')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

---
## Partie D : Feature scaling, PCA et visualisation

Le feature scaling est crucial pour les modèles basés sur les distances (SVM).
La PCA réduit la dimensionnalité pour la visualisation.

In [ ]:
# Impact du feature scaling

X_train_raw, X_test_raw, _, _ = train_test_split(
    X_bin, y_bin, test_size=0.3, random_state=42, stratify=y_bin
)

# Sans scaling
svm_no_scale = SVC(kernel='rbf', C=1.0)
svm_no_scale.fit(X_train_raw, y_bin[:len(X_train_raw)])
acc_no_scale = accuracy_score(
    y_bin[len(X_train_raw):],
    svm_no_scale.predict(X_test_raw)
)

# Avec scaling
svm_scale = SVC(kernel='rbf', C=1.0)
svm_scale.fit(X_train_scaled, y_train)
acc_scale = accuracy_score(y_test, svm_scale.predict(X_test_scaled))

print(f"Accuracy SVM RBF sans scaling : {acc_no_scale:.4f}")
print(f"Accuracy SVM RBF avec scaling : {acc_scale:.4f}")

In [ ]:
# PCA pour visualisation 2D

X_full = scaler.fit_transform(X)  # scaling sur toutes les données

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_full)

print(f"Variance expliquée par les 2 premières composantes : {pca.explained_variance_ratio_.sum():.3f}")
print(f"Composante 1 : {pca.explained_variance_ratio_[0]:.3f}")
print(f"Composante 2 : {pca.explained_variance_ratio_[1]:.3f}")

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolors='k', alpha=0.7)
plt.colorbar(scatter, ticks=[0, 1, 2], label='Classe')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('PCA — Iris Dataset (3 classes)')
plt.grid(alpha=0.3)
plt.show()

# Frontière de décision avec PCA 2D + SVM RBF
from sklearn.inspection import DecisionBoundaryDisplay

pca_model = SVC(kernel='rbf', C=1.0, gamma='scale')
pca_model.fit(X_pca[mask], y_bin)  # 2 classes uniquement

fig, ax = plt.subplots(figsize=(8, 6))
DecisionBoundaryDisplay.from_estimator(
    pca_model, X_pca[mask],
    response_method='predict',
    alpha=0.3,
    ax=ax,
    grid_resolution=100,
    cmap='coolwarm'
)
ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=y_bin, edgecolors='k', cmap='coolwarm')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Frontière de décision — SVM RBF après PCA')
plt.show()

---
## Exercices

### Exercice 1 : Wine dataset

Répétez l'analyse complète (SVM, RF, MLP, cross-validation, PCA) sur le dataset Wine.
Comparez les performances avec Iris.

In [ ]:
# Exercice 1 : Wine dataset
wine = datasets.load_wine()
print(f"Wine dataset : {wine.data.shape[0]} échantillons, {wine.data.shape[1]} features")
print(f"Classes : {wine.target_names}")

# TODO : Implémentez le pipeline complet pour Wine

### Exercice 2 : Impact du nombre d'arbres dans Random Forest

Étudiez l'impact du nombre d'estimateurs (10, 50, 100, 200, 500) sur la performance de Random Forest.

In [ ]:
# Exercice 2 : impact du nombre d'arbres
n_estimators_list = [10, 50, 100, 200, 500]
scores_rf = []

for n in n_estimators_list:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    cv_scores = cross_val_score(rf_temp, X_train_scaled, y_train, cv=5)
    scores_rf.append(cv_scores.mean())
    print(f"n_estimators={n:3d} : CV accuracy = {cv_scores.mean():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(n_estimators_list, scores_rf, 'o-', linewidth=2)
plt.xlabel('Nombre d\'estimateurs')
plt.ylabel('Accuracy CV (5-fold)')
plt.title('Impact du nombre d\'arbres — Random Forest')
plt.grid(alpha=0.3)
plt.show()

### Exercice 3 : Classification multi-classe (Iris 3 classes)

Reprenez tout le pipeline mais avec les 3 classes d'Iris. Comparez les métriques macro/micro F1.

In [ ]:
# Exercice 3 : classification multi-classe
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler_mc = StandardScaler()
X_train_mc_s = scaler_mc.fit_transform(X_train_mc)
X_test_mc_s = scaler_mc.transform(X_test_mc)

svm_mc = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
svm_mc.fit(X_train_mc_s, y_train_mc)
y_pred_mc = svm_mc.predict(X_test_mc_s)

print("Classification multi-classe (Iris, 3 classes) :")
print(classification_report(y_test_mc, y_pred_mc, target_names=iris.target_names))

---
## Pour aller plus loin

- Essayez d'autres kernels : laplacien, sigmoid, chi2
- Implémentez un pipeline complet avec `sklearn.pipeline.Pipeline`
- Comparez avec un Gradient Boosting (XGBoost, LightGBM)
- Références : [Bis06] Bishop — *Pattern Recognition and Machine Learning*, [MRT18] Mohri — *Foundations of Machine Learning*